# Project 1.2 — Structured Information Extraction

Deep Learning — A.A. 2025/2026  
Università degli Studi Roma Tre  

**Authors**  
Alfredo Carta  
Juan Miguel Pinos  

---

## Project Overview

This notebook implements a system for **Structured Information Extraction**, where a natural language input is converted into a **structured JSON representation**.

The task focuses on three main aspects:

- **Syntactic validity** of the generated JSON
- **Strict adherence** to the required schema (no missing or extra fields)
- **Reduction of hallucinations** in the extracted information

---

## Experimental Strategy

The experimental setup follows the methodology proposed in the project description:

- Use of **two Small Language Models (SLMs)** in the 3B–7B parameter range  
- **PRE fine-tuning evaluation** (zero-shot extraction)  
- **Lightweight fine-tuning using QLoRA in 4-bit precision**  
- **POST fine-tuning evaluation**  
- **Quantitative comparison between PRE and POST performance**

The notebook is optimized for **fast execution on Google Colab (T4 GPU)**.

---

## Generated Outputs

The experiments produce the following artifacts:

- `results.csv` — aggregated quantitative evaluation metrics  
- `examples.jsonl` — qualitative examples of model outputs  
- `report.md` — summary of the experimental results

In [ ]:
import sys
import subprocess
import time

PINNED = [
    "datasets==2.20.0",
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "peft==0.12.0",
    "bitsandbytes==0.43.3",
    "evaluate==0.4.2",
    "regex==2024.7.24",
]

def pip_install(packages):
    start = time.time()
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    print("Installing required packages...")
    subprocess.check_call(cmd)
    print(f"Installation completed in {time.time() - start:.1f} seconds.")

pip_install(PINNED)

print("If running on Colab, restart the runtime before executing the next cell.")

Installing required packages...
Installation completed in 41.9 seconds.
If running on Colab, restart the runtime before executing the next cell.


## Environment Setup and Hardware Check

In this section we import the main Python libraries required for the project and verify the availability of GPU resources.

Since the experiments involve running and fine-tuning Small Language Models (SLMs), access to a GPU significantly reduces execution time and enables efficient training using 4-bit quantization.

The notebook is designed to run on **Google Colab with a Tesla T4 GPU**.

We also fix the **random seed** to ensure experiment reproducibility and print basic information about the CUDA environment and available GPU memory.

In [ ]:
import torch
import random
import numpy as np
import gc
import os
import time

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"GPU total memory (GB): {total_mem:.2f}")
    print(f"GPU reserved (GB): {reserved:.2f}")
    print(f"GPU allocated (GB): {allocated:.2f}")
else:
    print("Warning: GPU not detected. This notebook is designed for GPU execution.")

CUDA available: True
GPU: Tesla T4
GPU total memory (GB): 15.64
GPU reserved (GB): 0.00
GPU allocated (GB): 0.00


## Dataset Preparation

For this project we use the **RecipeNLG dataset**, which contains cooking recipes described in natural language.  
Each recipe includes fields such as title, ingredients, and directions, which can be transformed into a structured representation.

To ensure **fast experimentation on a Colab Tesla T4 GPU**, we construct a small and reproducible subset of the dataset.

### Fast experimental configuration

- **Training set:** 300 samples  
- **Evaluation set:** 40 samples  
- **Fixed random seed:** ensures reproducibility across runs

The objective of this project is **not to maximize absolute performance**, but to demonstrate the **improvement obtained after fine-tuning**.

Therefore, we compare:

- **PRE fine-tuning performance** (zero-shot structured extraction)
- **POST fine-tuning performance** after lightweight QLoRA training

In [ ]:
Load dataset and create fast subset

DATASET_NAME = "SandhyaKilari/RecipeNLG_dataset"

print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split="train")

print("Total samples in dataset:", len(dataset))

# Fast configuration
TRAIN_SIZE = 300
EVAL_SIZE = 40

dataset = dataset.shuffle(seed=SEED)

train_dataset = dataset.select(range(TRAIN_SIZE))
eval_dataset = dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + EVAL_SIZE))

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

# Quick sanity check
print("\nExample sample:")
print("Title:", train_dataset[0]["title"])
print("Ingredients:", train_dataset[0]["ingredients"][:100])
print("Directions:", train_dataset[0]["directions"][:100])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total samples in dataset: 2231142
Train size: 300
Eval size: 40

Example sample:
Title: Kahlua Truffle
Ingredients: ["1 pkg. (large 5.1 oz.) instant vanilla pudding", "3 c. half and half", "6 c. angel food cake, cut 
Directions: ["Mix pudding with half and half; set aside.", "Sprinkle cake cubes evenly with Kahlua.", "Let stand


## Preprocessing and Silver Labels

In this section we implement the preprocessing pipeline used to prepare the data for structured information extraction.

The main steps include:

- Text cleaning for **ingredients** and **directions** fields.
- Lightweight **heuristic parsers** to extract:
  - cooking time
  - temperature
- Construction of **weak ground-truth labels (silver labels)**.

These labels follow the JSON schema used in the project:

{
  "ingredients": [string, ...],
  "cooking_time": integer (minutes) | null,
  "temperature": {"value": int, "unit": "C" | "F"} | null
}

Since the original dataset does not provide fully structured targets, we automatically derive approximate labels from the raw text fields.

Although **silver labels are not perfectly accurate**, they are sufficient for:

- fast automatic evaluation of model outputs
- lightweight fine-tuning (QLoRA)
- comparing **PRE vs POST fine-tuning performance**

In [ ]:
import re
import json
from typing import Any, Dict, List, Optional, Tuple

# -------------------------
# Text cleaning helpers
# -------------------------
_WS_RE = re.compile(r"\s+")
_BULLET_RE = re.compile(r"^[\-\*\u2022]\s+", re.MULTILINE)

def clean_text(s: str) -> str:
    s = s.strip()
    s = _BULLET_RE.sub("", s)
    s = _WS_RE.sub(" ", s)
    return s

def clean_ingredients(raw: str) -> List[str]:
    """
    RecipeNLG 'ingredients' is often a single string with separators.
    We split conservatively and keep short items if meaningful.
    """
    if raw is None:
        return []
    s = raw.replace("\r", "\n")
    # Common separators: comma, newline, semicolon
    parts = re.split(r"[,\n;]+", s)
    out = []
    for p in parts:
        p = clean_text(p)
        # Drop empty / ultra-short noise
        if len(p) < 2:
            continue
        out.append(p)
    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for x in out:
        key = x.lower()
        if key not in seen:
            seen.add(key)
            uniq.append(x)
    return uniq[:60]  # keep bounded

# -------------------------
# Time / temperature parsing (lightweight, heuristic)
# -------------------------
_TIME_PATTERNS = [
    # e.g., "bake for 20 minutes", "cook 1 hour", "simmer 45 min"
    re.compile(r"\b(?:for\s*)?(\d{1,3})\s*(?:minutes?|mins?)\b", re.IGNORECASE),
    re.compile(r"\b(?:for\s*)?(\d{1,2})\s*(?:hours?|hrs?)\b", re.IGNORECASE),
    # e.g., "1 hour 20 minutes"
    re.compile(r"\b(\d{1,2})\s*(?:hours?|hrs?)\s*(\d{1,3})\s*(?:minutes?|mins?)\b", re.IGNORECASE),
]

_TEMP_PATTERNS = [
    # Fahrenheit: "350 F", "350°F", "at 350 degrees"
    re.compile(r"\b(\d{2,3})\s*°?\s*F\b", re.IGNORECASE),
    re.compile(r"\b(\d{2,3})\s*degrees?\s*F\b", re.IGNORECASE),
    # Celsius: "180 C", "180°C"
    re.compile(r"\b(\d{2,3})\s*°?\s*C\b", re.IGNORECASE),
    re.compile(r"\b(\d{2,3})\s*degrees?\s*C\b", re.IGNORECASE),
]

def parse_cooking_time_minutes(text: str) -> Optional[int]:
    """
    Returns a single cooking time in minutes if a plausible mention is found.
    Heuristic: prefer 'hour+minutes' pattern, otherwise take the first minutes/hours.
    """
    t = text or ""
    t = t.replace("\n", " ")
    # hour+minutes
    m = _TIME_PATTERNS[2].search(t)
    if m:
        hours = int(m.group(1))
        mins = int(m.group(2))
        total = hours * 60 + mins
        if 1 <= total <= 24 * 60:
            return total

    # minutes
    m = _TIME_PATTERNS[0].search(t)
    if m:
        mins = int(m.group(1))
        if 1 <= mins <= 24 * 60:
            return mins

    # hours
    m = _TIME_PATTERNS[1].search(t)
    if m:
        hours = int(m.group(1))
        total = hours * 60
        if 1 <= total <= 24 * 60:
            return total

    return None

def parse_temperature(text: str) -> Optional[Dict[str, Any]]:
    """
    Returns {"value": int, "unit": "C"|"F"} if a plausible mention is found.
    Preference: explicit unit patterns.
    """
    t = text or ""
    t = t.replace("\n", " ")
    for pat in _TEMP_PATTERNS:
        m = pat.search(t)
        if not m:
            continue
        val = int(m.group(1))
        if pat.pattern.lower().find("c") != -1 and 30 <= val <= 300:
            return {"value": val, "unit": "C"}
        if pat.pattern.lower().find("f") != -1 and 100 <= val <= 600:
            return {"value": val, "unit": "F"}
    return None

# -------------------------
# Build model input + silver label
# -------------------------
def build_input_text(title: str, ingredients: str, directions: str) -> str:
    title = clean_text(title or "")
    ing_list = clean_ingredients(ingredients or "")
    ing_joined = "; ".join(ing_list[:30])
    directions = clean_text(directions or "")
    return f"Title: {title}\nIngredients: {ing_joined}\nDirections: {directions}"

def build_silver_label(sample: Dict[str, Any]) -> Dict[str, Any]:
    """
    JSON schema for the project:
    {
      "ingredients": [string, ...],
      "cooking_time": integer (minutes) or null,
      "temperature": {"value": int, "unit": "C"|"F"} or null
    }
    """
    ing = clean_ingredients(sample.get("ingredients", "") or "")
    directions = sample.get("directions", "") or ""
    cooking_time = parse_cooking_time_minutes(directions)
    temperature = parse_temperature(directions)

    return {
        "ingredients": ing,
        "cooking_time": cooking_time,
        "temperature": temperature,
    }

# Preview a couple of silver labels
for i in range(2):
    s = train_dataset[i]
    inp = build_input_text(s.get("title", ""), s.get("ingredients", ""), s.get("directions", ""))
    lab = build_silver_label(s)
    print(f"\n--- Sample {i} ---")
    print(inp[:350], "...")
    print("Silver label:", json.dumps(lab, ensure_ascii=False)[:300], "...")


--- Sample 0 ---
Title: Kahlua Truffle
Ingredients: ["1 pkg. (large 5.1 oz.) instant vanilla pudding"; "3 c. half and half"; "6 c. angel food cake; cut in cubes"; "2/3 to 3/4 c. Kahlua"; "1 pt. whipping cream"; "2 Tbsp. instant coffee"; "2 Tbsp. sugar"; "1 tsp. vanilla"; "1 bag Bits o' Brickle"]
Directions: ["Mix pudding with half and half; set aside.", "Sprinkle c ...
Silver label: {"ingredients": ["[\"1 pkg. (large 5.1 oz.) instant vanilla pudding\"", "\"3 c. half and half\"", "\"6 c. angel food cake", "cut in cubes\"", "\"2/3 to 3/4 c. Kahlua\"", "\"1 pt. whipping cream\"", "\"2 Tbsp. instant coffee\"", "\"2 Tbsp. sugar\"", "\"1 tsp. vanilla\"", "\"1 bag Bits o' Brickle\"]"] ...

--- Sample 1 ---
Title: Lo-Cal Tomato Juice Dressing
Ingredients: ["1/2 c. canned tomato juice"; "2 Tbsp. lemon juice or vinegar"; "1 tsp. salt"; "1 1/2 Tbsp. Worcestershire sauce"; "2 to 4 Tbsp. vegetable oil"; "1/2 tsp. dry mustard"; "1 tsp. minced onion"; "3 Tbsp. sugar substitute"]
Directions: ["Combin

## Fix Ingredient Parsing and Final Schema

In the RecipeNLG dataset the **ingredients** field is often stored as a
stringified Python list (e.g. `["1 cup sugar", "2 eggs"]`).

In this section we:

- Safely parse ingredient lists using `ast.literal_eval`.
- Normalize ingredient text.
- Define the **final JSON schema** used in the project.
- Build the final dataset composed of:

    - `input_text`
    - `silver_label` (JSON target)

This transformation prepares the dataset for both:

- **model evaluation**
- **QLoRA fine-tuning**

In [ ]:
import ast

def parse_ingredients_field(raw):
    """
    Convert stringified list into proper Python list safely.
    """
    if raw is None:
        return []

    if isinstance(raw, list):
        return raw

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [clean_text(str(x)) for x in parsed if len(str(x)) > 1]
    except Exception:
        pass

    # fallback
    return clean_ingredients(raw)

def build_input_and_label(sample):
    title = sample.get("title", "") or ""
    ingredients_raw = sample.get("ingredients", "") or ""
    directions = sample.get("directions", "") or ""

    ingredients_list = parse_ingredients_field(ingredients_raw)
    ingredients_list = ingredients_list[:50]

    input_text = (
        f"Title: {clean_text(title)}\n"
        f"Ingredients: {'; '.join(ingredients_list)}\n"
        f"Directions: {clean_text(directions)}"
    )

    cooking_time = parse_cooking_time_minutes(directions)
    temperature = parse_temperature(directions)

    silver_label = {
        "ingredients": ingredients_list,
        "cooking_time": cooking_time,
        "temperature": temperature,
    }

    return {
        "input_text": input_text,
        "label": silver_label
    }

# Apply transformation
train_processed = [build_input_and_label(s) for s in train_dataset]
eval_processed = [build_input_and_label(s) for s in eval_dataset]

# Quick check
for i in range(2):
    print("\n--- Sample", i, "---")
    print(train_processed[i]["input_text"][:300], "...")
    print("Label:", train_processed[i]["label"])


--- Sample 0 ---
Title: Kahlua Truffle
Ingredients: 1 pkg. (large 5.1 oz.) instant vanilla pudding; 3 c. half and half; 6 c. angel food cake, cut in cubes; 2/3 to 3/4 c. Kahlua; 1 pt. whipping cream; 2 Tbsp. instant coffee; 2 Tbsp. sugar; 1 tsp. vanilla; 1 bag Bits o' Brickle
Directions: ["Mix pudding with half and  ...
Label: {'ingredients': ['1 pkg. (large 5.1 oz.) instant vanilla pudding', '3 c. half and half', '6 c. angel food cake, cut in cubes', '2/3 to 3/4 c. Kahlua', '1 pt. whipping cream', '2 Tbsp. instant coffee', '2 Tbsp. sugar', '1 tsp. vanilla', "1 bag Bits o' Brickle"], 'cooking_time': 5, 'temperature': None}

--- Sample 1 ---
Title: Lo-Cal Tomato Juice Dressing
Ingredients: 1/2 c. canned tomato juice; 2 Tbsp. lemon juice or vinegar; 1 tsp. salt; 1 1/2 Tbsp. Worcestershire sauce; 2 to 4 Tbsp. vegetable oil; 1/2 tsp. dry mustard; 1 tsp. minced onion; 3 Tbsp. sugar substitute
Directions: ["Combine all ingredients and beat u ...
Label: {'ingredients': ['1/2 c. canned tomato

## JSON Schema Prompt and Robust Parsing

In this section we define the **prompting strategy and parsing utilities**
used to extract structured information from the model outputs.

The key components are:

- A **strict JSON schema** describing the expected output format.
- An **instructional prompt** that forces the model to return only valid JSON.
- **Robust parsing utilities** to extract JSON from the model output.
- **Normalization functions** to make evaluation more stable.

The final structure expected from the model is:

{
  "ingredients": [string, ...],
  "cooking_time": integer | null,
  "temperature": {"value": integer, "unit": "C" | "F"} | null
}

Because language models may produce additional text or formatting,
the parsing utilities automatically:

- extract JSON from code blocks or raw text
- validate the schema
- normalize values (e.g. convert `"20 minutes"` → `20`)
- remove invalid or implausible values

In [ ]:
import json
import re
from typing import Any, Dict, Optional, Tuple

JSON_SCHEMA_TEXT = """
Return ONLY a valid JSON object with EXACTLY these keys:
- "ingredients": an array of strings
- "cooking_time": an integer number of minutes, or null
- "temperature": an object {"value": integer, "unit": "C" or "F"}, or null

Rules:
- Do not add any extra keys.
- If a field is not specified in the text, use null (or [] for ingredients if none are present).
- Do not invent values. If unsure, use null.
"""

def build_prompt(input_text: str) -> str:
    return (
        "You are an information extraction system.\n"
        "Extract structured fields from the recipe text.\n\n"
        f"{JSON_SCHEMA_TEXT.strip()}\n\n"
        "Recipe text:\n"
        f"{input_text}\n\n"
        "JSON:\n"
    )

_CODEBLOCK_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL | re.IGNORECASE)

def extract_json_candidate(text: str) -> Optional[str]:
    """
    Try to extract a JSON object from model output.
    Preference order:
    1) JSON inside a fenced code block
    2) First {...} block in the text
    """
    if not text:
        return None

    m = _CODEBLOCK_RE.search(text)
    if m:
        return m.group(1).strip()

    # Fallback: find the first balanced-looking JSON object.
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start : end + 1].strip()

    return None

def coerce_to_int(x: Any) -> Optional[int]:
    if x is None:
        return None
    if isinstance(x, bool):
        return None
    if isinstance(x, int):
        return x
    if isinstance(x, float):
        if x.is_integer():
            return int(x)
        return None
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None
        # Extract first integer from string (e.g., "20 minutes")
        m = re.search(r"-?\d+", s)
        if m:
            try:
                return int(m.group(0))
            except Exception:
                return None
    return None

def normalize_ingredients(lst: Any) -> list:
    if not isinstance(lst, list):
        return []
    out = []
    seen = set()
    for item in lst:
        s = clean_text(str(item))
        if len(s) < 2:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out[:60]

def normalize_temperature(obj: Any) -> Optional[Dict[str, Any]]:
    if obj is None:
        return None
    if not isinstance(obj, dict):
        return None
    value = coerce_to_int(obj.get("value", None))
    unit = obj.get("unit", None)
    if isinstance(unit, str):
        unit = unit.strip().upper()
    else:
        unit = None

    if unit not in {"C", "F"}:
        return None
    if value is None:
        return None

    # Plausibility bounds
    if unit == "C" and not (30 <= value <= 300):
        return None
    if unit == "F" and not (100 <= value <= 600):
        return None

    return {"value": int(value), "unit": unit}

def validate_and_normalize_prediction(pred: Any) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Returns (normalized_pred, meta).
    meta contains:
      - valid_json: bool
      - schema_ok: bool
      - extra_keys: list
      - missing_keys: list
    """
    meta = {"valid_json": False, "schema_ok": False, "extra_keys": [], "missing_keys": []}

    if not isinstance(pred, dict):
        return {"ingredients": [], "cooking_time": None, "temperature": None}, meta

    meta["valid_json"] = True

    allowed = {"ingredients", "cooking_time", "temperature"}
    keys = set(pred.keys())
    meta["extra_keys"] = sorted(list(keys - allowed))
    meta["missing_keys"] = sorted(list(allowed - keys))
    meta["schema_ok"] = (len(meta["extra_keys"]) == 0 and len(meta["missing_keys"]) == 0)

    ingredients = normalize_ingredients(pred.get("ingredients", []))
    cooking_time = coerce_to_int(pred.get("cooking_time", None))
    if cooking_time is not None and not (1 <= cooking_time <= 24 * 60):
        cooking_time = None

    temperature = normalize_temperature(pred.get("temperature", None))

    return {"ingredients": ingredients, "cooking_time": cooking_time, "temperature": temperature}, meta

def parse_model_output_to_json(text: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Parse raw model text into a normalized prediction dict + meta.
    """
    candidate = extract_json_candidate(text)
    if candidate is None:
        return {"ingredients": [], "cooking_time": None, "temperature": None}, {
            "valid_json": False,
            "schema_ok": False,
            "extra_keys": [],
            "missing_keys": ["ingredients", "cooking_time", "temperature"],
            "error": "no_json_found",
        }

    try:
        obj = json.loads(candidate)
    except Exception as e:
        return {"ingredients": [], "cooking_time": None, "temperature": None}, {
            "valid_json": False,
            "schema_ok": False,
            "extra_keys": [],
            "missing_keys": ["ingredients", "cooking_time", "temperature"],
            "error": f"json_parse_error: {type(e).__name__}",
        }

    normalized, meta = validate_and_normalize_prediction(obj)
    return normalized, meta

# Quick sanity check on one example prompt
demo_prompt = build_prompt(train_processed[0]["input_text"])
print(demo_prompt[:600])

You are an information extraction system.
Extract structured fields from the recipe text.

Return ONLY a valid JSON object with EXACTLY these keys:
- "ingredients": an array of strings
- "cooking_time": an integer number of minutes, or null
- "temperature": an object {"value": integer, "unit": "C" or "F"}, or null

Rules:
- Do not add any extra keys.
- If a field is not specified in the text, use null (or [] for ingredients if none are present).
- Do not invent values. If unsure, use null.

Recipe text:
Title: Kahlua Truffle
Ingredients: 1 pkg. (large 5.1 oz.) instant vanilla pudding; 3 c. hal


# PRE evaluation (Qwen2.5-3B)

In this section we evaluate the base model **before fine-tuning**.

Steps:
1. Build prompt using the JSON schema
2. Generate model output
3. Parse JSON from the output
4. Normalize predictions
5. Compare predictions with silver labels

To keep runtime manageable we evaluate **up to 50 samples**.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID_3B = "Qwen/Qwen2.5-3B-Instruct"

# Hard cleanup (avoid VRAM fragmentation)
try:
    del model_3b
except Exception:
    pass
try:
    del tok_3b
except Exception:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

tok_3b = AutoTokenizer.from_pretrained(MODEL_ID_3B, use_fast=True)
if tok_3b.pad_token is None:
    tok_3b.pad_token = tok_3b.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_3B,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model_3b.config.use_cache = False
model_3b.eval()

print("Loaded:", MODEL_ID_3B)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
CUDA: True
GPU: Tesla T4


# Generation function + single example test (PRE)

This section implements the generation function used to query the model and perform a quick sanity check before the full PRE evaluation.

Generation is **deterministic** (`temperature=0`, `do_sample=False`) to reduce variability and enforce consistent JSON outputs.

Pipeline steps:
1. Generate model output from the prompt.
2. Extract JSON from the generated text.
3. Validate and normalize the JSON schema.
4. Compare the prediction with the silver label.

This single example ensures that **inference, parsing, and validation work correctly** before running the full evaluation.

In [ ]:
import torch

@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 192) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", padding=False, truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # Remove the prompt prefix if present (keeps only completion)
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text.strip()

# One-example PRE test
idx = 0
prompt = build_prompt(eval_processed[idx]["input_text"])
raw = generate_text(model_3b, tok_3b, prompt, max_new_tokens=192)

pred, meta = parse_model_output_to_json(raw)

print("RAW OUTPUT (first 800 chars):")
print(raw[:800])
print("\nPARSED PREDICTION:")
print(pred)
print("\nMETA:")
print(meta)
print("\nSILVER LABEL (for reference):")
print(eval_processed[idx]["label"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


RAW OUTPUT (first 800 chars):
{
  "ingredients": [
    "96 g oats",
    "32 g coconut flour",
    "2 tablespoons Splenda sugar substitute",
    "1 pinch salt",
    "(15 ounce) can white beans",
    "1 tablespoon hazelnut-flavored liquid coffee creamer",
    "7 tablespoons plain soymilk",
    "59 g peanut butter",
    "2 tablespoons cocoa powder",
    "12 teaspoon instant mocha-flavor coffee powder",
    "12 teaspoon cinnamon",
    "raisins"
  ],
  "cooking_time": null,
  "temperature": null
} Here is the structured data extracted from the provided recipe text:

```json
{
  "ingredients": [
    "96 g oats",
    "32 g coconut flour",
    "2 tablespoons Splenda sugar substitute",
    "1 pinch salt",
    "(15 ounce) can white beans",
    "1 tablespoon haz

PARSED PREDICTION:
{'ingredients': ['96 g oats', '32 g coconut flour', '2 tablespoons Splenda sugar substitute', '1 pinch salt', '(15 ounce) can white beans', '1 tablespoon hazelnut-flavored liquid coffee creamer', '7 tablespoons plain s

# PRE Evaluation on Subset (Qwen 3B)

In this section we run the **PRE fine-tuning evaluation** for the 3B model on a small evaluation subset.

The goal is to obtain quick baseline metrics before training.

Computed metrics:
- **valid_json_rate** — percentage of outputs that can be parsed as JSON
- **schema_ok_rate** — percentage of outputs matching the required schema (no missing or extra fields)
- **ingredient_f1** — F1 score on extracted ingredients (case-insensitive match)
- **cooking_time_acc** — accuracy of cooking time in minutes (ignoring `null`)
- **temperature_acc** — accuracy of temperature (value + unit)

A small number of **qualitative examples** (raw output, prediction, label) are also saved for the final report.

`max_new_tokens` is kept relatively low to speed up evaluation.

In [ ]:
from collections import defaultdict

def ingredient_f1(pred_list, gold_list):
    """
    Simple token-level set match on normalized strings (case-insensitive).
    """
    p = {clean_text(x).lower() for x in (pred_list or [])}
    g = {clean_text(x).lower() for x in (gold_list or [])}
    if len(p) == 0 and len(g) == 0:
        return 1.0
    if len(p) == 0 or len(g) == 0:
        return 0.0
    tp = len(p & g)
    prec = tp / max(1, len(p))
    rec = tp / max(1, len(g))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def cooking_time_acc(pred, gold):
    """
    Accuracy on cooking_time when at least one is non-null.
    If both null -> counted as correct.
    """
    p = pred.get("cooking_time", None)
    g = gold.get("cooking_time", None)
    if p is None and g is None:
        return 1.0
    if p is None or g is None:
        return 0.0
    return 1.0 if int(p) == int(g) else 0.0

def temperature_acc(pred, gold):
    """
    Accuracy on temperature when at least one is non-null.
    If both null -> correct.
    """
    p = pred.get("temperature", None)
    g = gold.get("temperature", None)
    if p is None and g is None:
        return 1.0
    if p is None or g is None:
        return 0.0
    return 1.0 if (p.get("value") == g.get("value") and p.get("unit") == g.get("unit")) else 0.0

def pre_evaluate_model(model, tokenizer, examples, max_new_tokens=192, tag="qwen3b_pre"):
    t0 = time.time()
    n = len(examples)

    metrics = defaultdict(float)
    debug_examples = []

    for i, ex in enumerate(examples):
        prompt = build_prompt(ex["input_text"])
        raw = generate_text(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        pred, meta = parse_model_output_to_json(raw)
        gold = ex["label"]

        metrics["valid_json_rate"] += 1.0 if meta.get("valid_json", False) else 0.0
        metrics["schema_ok_rate"] += 1.0 if meta.get("schema_ok", False) else 0.0
        metrics["ingredient_f1"] += ingredient_f1(pred.get("ingredients", []), gold.get("ingredients", []))
        metrics["cooking_time_acc"] += cooking_time_acc(pred, gold)
        metrics["temperature_acc"] += temperature_acc(pred, gold)

        # Save a few qualitative examples
        if i < 8:
            debug_examples.append({
                "idx": i,
                "tag": tag,
                "input_text": ex["input_text"],
                "raw_output": raw,
                "prediction": pred,
                "meta": meta,
                "label": gold,
            })

        # Progress + ETA
        if (i + 1) % 5 == 0 or (i + 1) == n:
            elapsed = time.time() - t0
            rate = elapsed / (i + 1)
            eta = rate * (n - (i + 1))
            print(f"[{tag}] {i+1}/{n} done | elapsed {elapsed:.1f}s | ETA {eta:.1f}s")

    # Average metrics
    for k in list(metrics.keys()):
        metrics[k] /= max(1, n)

    return dict(metrics), debug_examples

qwen3b_pre_metrics, qwen3b_pre_examples = pre_evaluate_model(
    model_3b,
    tok_3b,
    eval_processed,
    max_new_tokens=192,
    tag="qwen3b_pre"
)

print("\nQwen 3B PRE metrics:")
for k, v in qwen3b_pre_metrics.items():
    print(f"- {k}: {v:.4f}")

print("\nSaved qualitative examples:", len(qwen3b_pre_examples))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[qwen3b_pre] 5/40 done | elapsed 40.7s | ETA 285.2s
[qwen3b_pre] 10/40 done | elapsed 82.0s | ETA 245.9s
[qwen3b_pre] 15/40 done | elapsed 121.1s | ETA 201.8s
[qwen3b_pre] 20/40 done | elapsed 160.3s | ETA 160.3s
[qwen3b_pre] 25/40 done | elapsed 199.8s | ETA 119.9s
[qwen3b_pre] 30/40 done | elapsed 238.6s | ETA 79.5s
[qwen3b_pre] 35/40 done | elapsed 277.9s | ETA 39.7s
[qwen3b_pre] 40/40 done | elapsed 317.0s | ETA 0.0s

Qwen 3B PRE metrics:
- valid_json_rate: 1.0000
- schema_ok_rate: 1.0000
- ingredient_f1: 0.7159
- cooking_time_acc: 0.6000
- temperature_acc: 0.5750

Saved qualitative examples: 8


# Build SFT Dataset for Qwen 3B

In this section we build the dataset for **supervised fine-tuning (SFT)** of the 3B model.

Each training example contains:
- **Prompt**: the structured prompt including the JSON schema and the recipe text.
- **Target**: the corresponding **silver label JSON**, with no additional text.

For causal language model training:
- The final sequence is **prompt + target JSON**.
- All prompt tokens are **masked** (`labels = -100`).
- The loss is computed **only on the target JSON tokens**.

To keep training efficient:
- the **sequence length is limited**,
- the **training set is small** (~300 examples),
- the number of training steps will be limited in the next section.

Finally, the dataset is tokenized and converted into the format required for model training.

In [ ]:
from typing import Dict, List
import json

MAX_SEQ_LEN = 640  # keep small for speed on T4

def label_to_json_string(label: Dict[str, Any]) -> str:
    """
    Serialize silver label as strict JSON (no extra text).
    """
    return json.dumps(label, ensure_ascii=False)

def format_sft_example(ex: Dict[str, Any]) -> Dict[str, str]:
    prompt = build_prompt(ex["input_text"])
    target = label_to_json_string(ex["label"])
    return {"prompt": prompt, "target": target}

train_sft = [format_sft_example(x) for x in train_processed]
eval_sft = [format_sft_example(x) for x in eval_processed]

print("Train SFT examples:", len(train_sft))
print("Eval SFT examples:", len(eval_sft))

# Tokenization + masking (labels = -100 for prompt tokens)
def tokenize_sft(batch: Dict[str, List[str]], tokenizer):
    prompts = batch["prompt"]
    targets = batch["target"]

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for p, t in zip(prompts, targets):
        p_ids = tokenizer(p, add_special_tokens=False).input_ids
        t_ids = tokenizer(t, add_special_tokens=False).input_ids + [tokenizer.eos_token_id]

        # Truncate from the left of the prompt if needed (keep target intact as much as possible)
        total_len = len(p_ids) + len(t_ids)
        if total_len > MAX_SEQ_LEN:
            overflow = total_len - MAX_SEQ_LEN
            p_ids = p_ids[max(0, overflow):]

        input_ids = p_ids + t_ids
        attention_mask = [1] * len(input_ids)
        labels = [-100] * len(p_ids) + t_ids[:]  # learn only the JSON

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

from datasets import Dataset

train_sft_ds = Dataset.from_list(train_sft).map(lambda b: tokenize_sft(b, tok_3b), batched=True, remove_columns=["prompt", "target"])
eval_sft_ds = Dataset.from_list(eval_sft).map(lambda b: tokenize_sft(b, tok_3b), batched=True, remove_columns=["prompt", "target"])

print("Tokenized train columns:", train_sft_ds.column_names)
print("Tokenized eval columns:", eval_sft_ds.column_names)

# Quick sanity check (decode target portion only)
sample = train_sft_ds[0]
labels = sample["labels"]
toks = [x for x in labels if x != -100]
print("\nDecoded target JSON (first 400 chars):")
print(tok_3b.decode(toks[:400], skip_special_tokens=True))

Train SFT examples: 300
Eval SFT examples: 40


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenized train columns: ['input_ids', 'attention_mask', 'labels']
Tokenized eval columns: ['input_ids', 'attention_mask', 'labels']

Decoded target JSON (first 400 chars):
{"ingredients": ["1 pkg. (large 5.1 oz.) instant vanilla pudding", "3 c. half and half", "6 c. angel food cake, cut in cubes", "2/3 to 3/4 c. Kahlua", "1 pt. whipping cream", "2 Tbsp. instant coffee", "2 Tbsp. sugar", "1 tsp. vanilla", "1 bag Bits o' Brickle"], "cooking_time": 5, "temperature": null}


# QLoRA Fine-Tuning (Qwen 3B)

In this section we perform **fine-tuning of the Qwen 3B model** using **QLoRA with 4-bit quantization**.

Goal:
- obtain a measurable **POST fine-tuning improvement over the PRE baseline**
- keep the **training time very short** for execution on a **T4 GPU**

Strategy:
- reload the base model in **4-bit quantization**
- apply **LoRA adapters** to the attention layers
- run a short training controlled by **`max_steps`**

Fast configuration:
- limited **`max_steps`** (default **120**)
- small batch size with **gradient accumulation**
- effectively **one epoch**, controlled by `max_steps`
- **reduced sequence length** to limit VRAM usage

During training, an **ETA** is periodically printed to monitor execution time.

Output of this section:
- LoRA adapter saved on disk in **`qwen3b_adapter`**

In [ ]:
import math
import torch
from transformers import BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

QWEN_ADAPTER_DIR = "qwen3b_adapter"

# -------------------------
# Reload base model in 4-bit for training
# -------------------------
try:
    del model_3b
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model_3b = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_3B,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model_3b.config.use_cache = False

model_3b = prepare_model_for_kbit_training(model_3b)

# LoRA config (small for speed)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # common safe target list
)

model_3b = get_peft_model(model_3b, lora_config)
model_3b.print_trainable_parameters()

# -------------------------
# Training args (FAST)
# -------------------------
MAX_STEPS = 120
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LR = 2e-4

args = TrainingArguments(
    output_dir="qwen3b_qlora_out",
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=60,
    save_total_limit=1,
    evaluation_strategy="no",
    report_to="none",
    fp16=True,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
)

data_collator = DataCollatorForLanguageModeling(tok_3b, mlm=False)

# -------------------------
# Custom Trainer to print ETA
# -------------------------
class ETATrainer(Trainer):
    def __init__(self, *trainer_args, **trainer_kwargs):
        super().__init__(*trainer_args, **trainer_kwargs)
        self._t0 = None

    def training_step(self, model, inputs):
        if self._t0 is None:
            self._t0 = time.time()

        step = self.state.global_step
        out = super().training_step(model, inputs)

        # Print ETA every 10 steps (aligned with logging_steps)
        if (step + 1) % 10 == 0:
            elapsed = time.time() - self._t0
            done = step + 1
            total = self.args.max_steps
            rate = elapsed / max(1, done)
            eta = rate * max(0, total - done)
            print(f"[qwen3b_train] step {done}/{total} | elapsed {elapsed:.1f}s | ETA {eta:.1f}s")

        return out

trainer = ETATrainer(
    model=model_3b,
    args=args,
    train_dataset=train_sft_ds,
    tokenizer=tok_3b,
    data_collator=data_collator,
)

train_result = trainer.train()

# Save adapter
trainer.model.save_pretrained(QWEN_ADAPTER_DIR)
tok_3b.save_pretrained(QWEN_ADAPTER_DIR)

print("\nSaved Qwen 3B adapter to:", QWEN_ADAPTER_DIR)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
10,1.856100
20,1.275200
30,0.908900
40,0.763800
50,0.765400
60,0.811300
70,0.693700
80,0.725600
90,0.763600
100,0.668700


[qwen3b_train] step 10/120 | elapsed 94.4s | ETA 1038.1s
[qwen3b_train] step 10/120 | elapsed 95.4s | ETA 1049.3s
[qwen3b_train] step 10/120 | elapsed 96.6s | ETA 1062.2s
[qwen3b_train] step 10/120 | elapsed 97.8s | ETA 1075.6s
[qwen3b_train] step 10/120 | elapsed 99.0s | ETA 1088.9s
[qwen3b_train] step 10/120 | elapsed 100.5s | ETA 1105.7s
[qwen3b_train] step 10/120 | elapsed 102.2s | ETA 1124.1s
[qwen3b_train] step 10/120 | elapsed 103.6s | ETA 1139.8s
[qwen3b_train] step 20/120 | elapsed 202.7s | ETA 1013.5s
[qwen3b_train] step 20/120 | elapsed 203.8s | ETA 1018.9s
[qwen3b_train] step 20/120 | elapsed 205.2s | ETA 1026.0s
[qwen3b_train] step 20/120 | elapsed 206.4s | ETA 1032.0s
[qwen3b_train] step 20/120 | elapsed 207.7s | ETA 1038.5s
[qwen3b_train] step 20/120 | elapsed 209.1s | ETA 1045.7s
[qwen3b_train] step 20/120 | elapsed 210.3s | ETA 1051.3s
[qwen3b_train] step 20/120 | elapsed 211.6s | ETA 1057.9s
[qwen3b_train] step 30/120 | elapsed 306.8s | ETA 920.3s
[qwen3b_train] step 

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


[qwen3b_train] step 70/120 | elapsed 728.8s | ETA 520.6s
[qwen3b_train] step 70/120 | elapsed 730.1s | ETA 521.5s
[qwen3b_train] step 70/120 | elapsed 731.4s | ETA 522.4s
[qwen3b_train] step 70/120 | elapsed 732.6s | ETA 523.3s
[qwen3b_train] step 70/120 | elapsed 734.0s | ETA 524.3s
[qwen3b_train] step 70/120 | elapsed 735.5s | ETA 525.4s
[qwen3b_train] step 70/120 | elapsed 736.7s | ETA 526.2s
[qwen3b_train] step 70/120 | elapsed 738.2s | ETA 527.3s
[qwen3b_train] step 80/120 | elapsed 835.1s | ETA 417.5s
[qwen3b_train] step 80/120 | elapsed 836.2s | ETA 418.1s
[qwen3b_train] step 80/120 | elapsed 837.4s | ETA 418.7s
[qwen3b_train] step 80/120 | elapsed 839.0s | ETA 419.5s
[qwen3b_train] step 80/120 | elapsed 840.7s | ETA 420.3s
[qwen3b_train] step 80/120 | elapsed 842.3s | ETA 421.1s
[qwen3b_train] step 80/120 | elapsed 843.9s | ETA 421.9s
[qwen3b_train] step 80/120 | elapsed 845.2s | ETA 422.6s
[qwen3b_train] step 90/120 | elapsed 941.7s | ETA 313.9s
[qwen3b_train] step 90/120 | el

# POST evaluation (Qwen 3B fine-tuned) and PRE vs POST comparison

In this section we evaluate the **fine-tuned Qwen 3B model** and compare its performance with the baseline obtained before fine-tuning.

The procedure is the following:

- Load the **Qwen 3B base model** in **4-bit quantization**.
- Attach the **fine-tuned PEFT adapter** produced during training.
- Run the **POST evaluation** on the same evaluation dataset used in the PRE phase.
- Compare **PRE vs POST metrics** to quantify the improvement obtained through fine-tuning.

Important:

The evaluation uses the **same generation function and the same metrics** employed during the PRE evaluation to ensure a **fair and reproducible comparison**.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# -------------------------
# Config
# -------------------------
BASE_MODEL_ID = MODEL_ID_3B
ADAPTER_DIR = "qwen3b_adapter"

# -------------------------
# Hard cleanup (free VRAM)
# -------------------------
try:
    del model_3b
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# -------------------------
# Load base model in 4-bit
# -------------------------
tok_post = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tok_post.pad_token is None:
    tok_post.pad_token = tok_post.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model.eval()

# -------------------------
# Attach adapter
# -------------------------
model_post = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_post.eval()

print("Loaded base:", BASE_MODEL_ID)
print("Loaded adapter:", ADAPTER_DIR)

# -------------------------
# Run POST evaluation
# -------------------------
qwen3b_post_metrics, qwen3b_post_examples = pre_evaluate_model(
    model_post,
    tok_post,
    eval_processed,
    max_new_tokens=192,
    tag="qwen3b_post"
)

print("\nQwen 3B POST metrics:")
for k, v in qwen3b_post_metrics.items():
    print(f"- {k}: {v:.4f}")

# -------------------------
# Compare PRE vs POST
# -------------------------
print("\nQwen 3B PRE vs POST (delta):")
for k in qwen3b_pre_metrics.keys():
    pre_v = qwen3b_pre_metrics[k]
    post_v = qwen3b_post_metrics[k]
    print(f"- {k}: PRE {pre_v:.4f} -> POST {post_v:.4f} | Δ {post_v - pre_v:+.4f}")

print("\nSaved qualitative POST examples:", len(qwen3b_post_examples))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded base: Qwen/Qwen2.5-3B-Instruct
Loaded adapter: qwen3b_adapter


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[qwen3b_post] 5/40 done | elapsed 77.0s | ETA 538.8s
[qwen3b_post] 10/40 done | elapsed 125.9s | ETA 377.6s
[qwen3b_post] 15/40 done | elapsed 175.5s | ETA 292.5s
[qwen3b_post] 20/40 done | elapsed 226.9s | ETA 226.9s
[qwen3b_post] 25/40 done | elapsed 275.2s | ETA 165.1s
[qwen3b_post] 30/40 done | elapsed 325.2s | ETA 108.4s
[qwen3b_post] 35/40 done | elapsed 383.8s | ETA 54.8s
[qwen3b_post] 40/40 done | elapsed 441.6s | ETA 0.0s

Qwen 3B POST metrics:
- valid_json_rate: 1.0000
- schema_ok_rate: 1.0000
- ingredient_f1: 0.9583
- cooking_time_acc: 0.6500
- temperature_acc: 0.9250

Qwen 3B PRE vs POST (delta):
- valid_json_rate: PRE 1.0000 -> POST 1.0000 | Δ +0.0000
- schema_ok_rate: PRE 1.0000 -> POST 1.0000 | Δ +0.0000
- ingredient_f1: PRE 0.7159 -> POST 0.9583 | Δ +0.2425
- cooking_time_acc: PRE 0.6000 -> POST 0.6500 | Δ +0.0500
- temperature_acc: PRE 0.5750 -> POST 0.9250 | Δ +0.3500

Saved qualitative POST examples: 8


# Second model (7B): Mistral 7B PRE → QLoRA FAST → POST

After evaluating Qwen 3B, we run the same experimental protocol on a second Small Language Model (SLM).

Model:
- mistralai/Mistral-7B-Instruct-v0.3

Goal:
- Replicate the same pipeline used for Qwen 3B:
  1. PRE evaluation (baseline model)
  2. QLoRA fine-tuning
  3. POST evaluation
  4. Quantitative comparison PRE vs POST

To keep the experiment lightweight:
- the model will be loaded in **4-bit quantization**
- fine-tuning will use **QLoRA**
- training will run with **reduced max_steps**

This cell only defines the model constants and a cleanup helper to prevent GPU memory issues before loading the 7B model.

In [ ]:
import gc
import torch

MODEL_ID_7B = "mistralai/Mistral-7B-Instruct-v0.3"
M7B_ADAPTER_DIR = "mistral7b_adapter"

def hard_cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print("Second model:", MODEL_ID_7B)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Second model: mistralai/Mistral-7B-Instruct-v0.3
CUDA: True
GPU: Tesla T4


# Load Mistral 7B in 4-bit (PRE evaluation)

In this step we load the second model used in the project, **Mistral-7B-Instruct**, for the **pre fine-tuning evaluation** phase.

Since the model has around **7 billion parameters**, we load it using **4-bit NF4 quantization (bitsandbytes)**. This significantly reduces the GPU memory footprint, allowing the model to run on the **Tesla T4 GPU available in Google Colab** while still maintaining good inference quality.

The **same evaluation pipeline used for Qwen 2.5-3B** is reused here in order to ensure a fair comparison between models. In particular, we keep:
- the same prompt template,
- the same JSON schema for structured extraction,
- the same parsing logic,
- and the same evaluation metrics.

This guarantees that the results obtained with the two models are directly comparable in the **PRE fine-tuning benchmark**.

Note: `use_cache = False` is set to ensure compatibility with the training phase and to avoid issues during gradient checkpointing in later steps.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Hard cleanup (avoid OOM)
try:
    del model_post
except Exception:
    pass
try:
    del base_model
except Exception:
    pass
try:
    del tok_post
except Exception:
    pass

hard_cleanup()

tok_7b = AutoTokenizer.from_pretrained(MODEL_ID_7B, use_fast=True)
if tok_7b.pad_token is None:
    tok_7b.pad_token = tok_7b.eos_token

bnb_config_7b = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model_7b = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_7B,
    device_map="auto",
    quantization_config=bnb_config_7b,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model_7b.config.use_cache = False
model_7b.eval()

print("Loaded:", MODEL_ID_7B)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.3


# PRE Evaluation on eval_processed (Mistral 7B)

We run the **PRE fine-tuning evaluation** for the 7B model on the `eval_processed` subset.

The evaluation setup is identical to the one used for **Qwen 2.5-3B** to ensure a fair comparison between models:

- same prompt template with JSON schema  
- same parsing and validation logic  
- same evaluation metrics  

During the evaluation we also collect **qualitative examples** that will be included in the final report.

Progress information and an **ETA estimate** are printed during execution.

In [ ]:
m7b_pre_metrics, m7b_pre_examples = pre_evaluate_model(
    model_7b,
    tok_7b,
    eval_processed,
    max_new_tokens=192,
    tag="mistral7b_pre"
)

print("\nMistral 7B PRE metrics:")
for k, v in m7b_pre_metrics.items():
    print(f"- {k}: {v:.4f}")

print("\nSaved qualitative examples:", len(m7b_pre_examples))

[mistral7b_pre] 5/40 done | elapsed 61.9s | ETA 433.4s
[mistral7b_pre] 10/40 done | elapsed 105.7s | ETA 317.2s
[mistral7b_pre] 15/40 done | elapsed 153.3s | ETA 255.5s
[mistral7b_pre] 20/40 done | elapsed 199.6s | ETA 199.6s
[mistral7b_pre] 25/40 done | elapsed 240.6s | ETA 144.4s
[mistral7b_pre] 30/40 done | elapsed 288.4s | ETA 96.1s
[mistral7b_pre] 35/40 done | elapsed 339.9s | ETA 48.6s
[mistral7b_pre] 40/40 done | elapsed 391.0s | ETA 0.0s

Mistral 7B PRE metrics:
- valid_json_rate: 1.0000
- schema_ok_rate: 1.0000
- ingredient_f1: 0.9079
- cooking_time_acc: 0.5000
- temperature_acc: 0.6750

Saved qualitative examples: 8


# Build SFT Dataset for Mistral 7B (Tokenization + Prompt Masking)

We now build the **Supervised Fine-Tuning (SFT) dataset** for **Mistral 7B**, using the same procedure previously applied to **Qwen 2.5–3B**.

Each training example is composed of:

- a **prompt** containing the instruction and input text  
- a **target** containing the structured JSON output  

During tokenization:

- the **prompt tokens are masked** with `-100`
- the **loss is computed only on the JSON target tokens**

This ensures that the model learns to generate the **structured JSON output** without being penalized for reproducing the prompt.

Note:
We reuse the same examples (`train_sft` and `eval_sft`), but they are **re-tokenized using the Mistral tokenizer (`tok_7b`)**.

In [ ]:
# Build tokenized SFT dataset for Mistral 7B (prompt masking)

from datasets import Dataset

MAX_SEQ_LEN_7B = 640  # keep bounded for T4 speed

def tokenize_sft_with_tok(batch, tokenizer, max_seq_len: int):
    prompts = batch["prompt"]
    targets = batch["target"]

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for p, t in zip(prompts, targets):
        p_ids = tokenizer(p, add_special_tokens=False).input_ids
        t_ids = tokenizer(t, add_special_tokens=False).input_ids + [tokenizer.eos_token_id]

        total_len = len(p_ids) + len(t_ids)
        if total_len > max_seq_len:
            overflow = total_len - max_seq_len
            p_ids = p_ids[max(0, overflow):]

        input_ids = p_ids + t_ids
        attention_mask = [1] * len(input_ids)
        labels = [-100] * len(p_ids) + t_ids[:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

train_sft_ds_7b = Dataset.from_list(train_sft).map(
    lambda b: tokenize_sft_with_tok(b, tok_7b, MAX_SEQ_LEN_7B),
    batched=True,
    remove_columns=["prompt", "target"],
)

eval_sft_ds_7b = Dataset.from_list(eval_sft).map(
    lambda b: tokenize_sft_with_tok(b, tok_7b, MAX_SEQ_LEN_7B),
    batched=True,
    remove_columns=["prompt", "target"],
)

print("Tokenized train columns:", train_sft_ds_7b.column_names)
print("Tokenized eval columns:", eval_sft_ds_7b.column_names)

# sanity check: decode target tokens only
sample = train_sft_ds_7b[0]
toks = [x for x in sample["labels"] if x != -100]
print("\nDecoded target JSON (first 400 chars):")
print(tok_7b.decode(toks[:400], skip_special_tokens=True))

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenized train columns: ['input_ids', 'attention_mask', 'labels']
Tokenized eval columns: ['input_ids', 'attention_mask', 'labels']

Decoded target JSON (first 400 chars):
{"ingredients": ["1 pkg. (large 5.1 oz.) instant vanilla pudding", "3 c. half and half", "6 c. angel food cake, cut in cubes", "2/3 to 3/4 c. Kahlua", "1 pt. whipping cream", "2 Tbsp. instant coffee", "2 Tbsp. sugar", "1 tsp. vanilla", "1 bag Bits o' Brickle"], "cooking_time": 5, "temperature": null}


# QLoRA Fine-Tuning for Mistral 7B (Fast Configuration)

We fine-tune **Mistral 7B** using **QLoRA with 4-bit quantization**, allowing the model to train efficiently within the memory limits of a **Tesla T4 GPU**.

To keep the experiment lightweight and reproducible, we use a **fast configuration**:

- reduced number of training steps (`max_steps`)
- small batch size with **gradient accumulation**
- limited sequence length to control memory usage

This setup is sufficient to observe the **impact of supervised fine-tuning** while keeping training time manageable.

Key aspects of the configuration:

- **QLoRA (4-bit NF4)** for memory-efficient training  
- **LoRA adapters** applied to the attention projection layers  
- **Paged AdamW 8-bit optimizer** for efficient optimization  
- **Custom trainer with ETA logging** to monitor training progress

Output:
The trained adapter is saved to:

`mistral7b_adapter`

In [ ]:
import torch
from transformers import BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# -------------------------
# Reload base model in 4-bit for training
# -------------------------
try:
    del model_7b
except Exception:
    pass
hard_cleanup()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model_7b = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_7B,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model_7b.config.use_cache = False
model_7b = prepare_model_for_kbit_training(model_7b)

# LoRA config (fast + stable)
lora_config_7b = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model_7b = get_peft_model(model_7b, lora_config_7b)
model_7b.print_trainable_parameters()

# -------------------------
# Training args (FAST)
# -------------------------
MAX_STEPS_7B = 80
PER_DEVICE_BATCH_7B = 1
GRAD_ACCUM_7B = 8
LR_7B = 2e-4

args_7b = TrainingArguments(
    output_dir="mistral7b_qlora_out",
    per_device_train_batch_size=PER_DEVICE_BATCH_7B,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_7B,
    learning_rate=LR_7B,
    max_steps=MAX_STEPS_7B,
    logging_steps=10,
    save_steps=40,
    save_total_limit=1,
    evaluation_strategy="no",
    report_to="none",
    fp16=True,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
)

data_collator_7b = DataCollatorForLanguageModeling(tok_7b, mlm=False)

# -------------------------
# Custom Trainer with ETA
# -------------------------
class ETATrainer7B(Trainer):
    def __init__(self, *trainer_args, **trainer_kwargs):
        super().__init__(*trainer_args, **trainer_kwargs)
        self._t0 = None
        self._last_print_step = -1

    def training_step(self, model, inputs):
        if self._t0 is None:
            self._t0 = time.time()

        out = super().training_step(model, inputs)

        step = self.state.global_step
        # Print only once per 10 steps
        if (step + 1) % 10 == 0 and step != self._last_print_step:
            self._last_print_step = step
            elapsed = time.time() - self._t0
            done = step + 1
            total = self.args.max_steps
            rate = elapsed / max(1, done)
            eta = rate * max(0, total - done)
            print(f"[mistral7b_train] step {done}/{total} | elapsed {elapsed:.1f}s | ETA {eta:.1f}s")

        return out

trainer_7b = ETATrainer7B(
    model=model_7b,
    args=args_7b,
    train_dataset=train_sft_ds_7b,
    tokenizer=tok_7b,
    data_collator=data_collator_7b,
)

trainer_7b.train()

trainer_7b.model.save_pretrained(M7B_ADAPTER_DIR)
tok_7b.save_pretrained(M7B_ADAPTER_DIR)

print("\nSaved Mistral 7B adapter to:", M7B_ADAPTER_DIR)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


trainable params: 6,815,744 || all params: 7,254,839,296 || trainable%: 0.0939


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
10,1.033400
20,0.597700
30,0.574300
40,0.542400
50,0.535500
60,0.564700
70,0.477200
80,0.511600


[mistral7b_train] step 10/80 | elapsed 227.6s | ETA 1593.1s
[mistral7b_train] step 20/80 | elapsed 486.1s | ETA 1458.4s
[mistral7b_train] step 30/80 | elapsed 735.4s | ETA 1225.6s
[mistral7b_train] step 40/80 | elapsed 991.4s | ETA 991.4s


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


[mistral7b_train] step 50/80 | elapsed 1244.4s | ETA 746.6s
[mistral7b_train] step 60/80 | elapsed 1500.4s | ETA 500.1s
[mistral7b_train] step 70/80 | elapsed 1751.6s | ETA 250.2s
[mistral7b_train] step 80/80 | elapsed 2004.7s | ETA 0.0s

Saved Mistral 7B adapter to: mistral7b_adapter


# POST Evaluation (Mistral 7B Fine-Tuned) and PRE vs POST Comparison

In this section we evaluate the **fine-tuned Mistral 7B model** and compare its performance with the **baseline (PRE) results**.

Steps performed:

1. Load the **Mistral 7B base model** in 4-bit quantization.
2. Load and apply the **fine-tuned LoRA adapter**.
3. Run the **POST evaluation** on the same evaluation set used for the baseline.
4. Compute and print the **PRE vs POST comparison**, highlighting the performance improvement (delta).

We also store **qualitative examples** generated by the fine-tuned model to inspect improvements in structured JSON extraction.

Output:
- POST evaluation metrics
- PRE vs POST metric comparison
- qualitative generation examples

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Cleanup training model to free VRAM
try:
    del model_7b
except Exception:
    pass
hard_cleanup()

BASE_MODEL_ID_7B = MODEL_ID_7B
ADAPTER_DIR_7B = "mistral7b_adapter"

tok_post_7b = AutoTokenizer.from_pretrained(BASE_MODEL_ID_7B, use_fast=True)
if tok_post_7b.pad_token is None:
    tok_post_7b.pad_token = tok_post_7b.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_7b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID_7B,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
base_7b.config.use_cache = False
base_7b.eval()

model_post_7b = PeftModel.from_pretrained(base_7b, ADAPTER_DIR_7B)
model_post_7b.eval()

print("Loaded base:", BASE_MODEL_ID_7B)
print("Loaded adapter:", ADAPTER_DIR_7B)

m7b_post_metrics, m7b_post_examples = pre_evaluate_model(
    model_post_7b,
    tok_post_7b,
    eval_processed,
    max_new_tokens=192,
    tag="mistral7b_post"
)

print("\nMistral 7B POST metrics:")
for k, v in m7b_post_metrics.items():
    print(f"- {k}: {v:.4f}")

print("\nMistral 7B PRE vs POST (delta):")
for k in m7b_pre_metrics.keys():
    pre_v = m7b_pre_metrics[k]
    post_v = m7b_post_metrics[k]
    print(f"- {k}: PRE {pre_v:.4f} -> POST {post_v:.4f} | Δ {post_v - pre_v:+.4f}")

print("\nSaved qualitative POST examples:", len(m7b_post_examples))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded base: mistralai/Mistral-7B-Instruct-v0.3
Loaded adapter: mistral7b_adapter


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[mistral7b_post] 5/40 done | elapsed 81.3s | ETA 569.1s
[mistral7b_post] 10/40 done | elapsed 135.2s | ETA 405.5s
[mistral7b_post] 15/40 done | elapsed 190.2s | ETA 317.1s
[mistral7b_post] 20/40 done | elapsed 247.5s | ETA 247.5s
[mistral7b_post] 25/40 done | elapsed 300.8s | ETA 180.5s
[mistral7b_post] 30/40 done | elapsed 357.7s | ETA 119.2s
[mistral7b_post] 35/40 done | elapsed 423.1s | ETA 60.4s
[mistral7b_post] 40/40 done | elapsed 486.5s | ETA 0.0s

Mistral 7B POST metrics:
- valid_json_rate: 0.9750
- schema_ok_rate: 0.9750
- ingredient_f1: 0.9750
- cooking_time_acc: 0.8750
- temperature_acc: 0.9250

Mistral 7B PRE vs POST (delta):
- valid_json_rate: PRE 1.0000 -> POST 0.9750 | Δ -0.0250
- schema_ok_rate: PRE 1.0000 -> POST 0.9750 | Δ -0.0250
- ingredient_f1: PRE 0.9079 -> POST 0.9750 | Δ +0.0671
- cooking_time_acc: PRE 0.5000 -> POST 0.8750 | Δ +0.3750
- temperature_acc: PRE 0.6750 -> POST 0.9250 | Δ +0.2500

Saved qualitative POST examples: 8


# Saving Final Results: results.csv + examples.jsonl + report.md

In this section we generate the **final artifacts required by the project**.

Three files are produced:

1. **results.csv**  
   Contains the evaluation metrics for both models (**Qwen 3B** and **Mistral 7B**) before and after fine-tuning:
   - PRE metrics
   - POST metrics
   - performance deltas (POST − PRE)

2. **examples.jsonl**  
   Stores **qualitative examples** of model generations for inspection.  
   For each example we save:
   - input text
   - raw model output
   - parsed prediction
   - ground-truth label
   - optional metadata

   Examples are collected for **PRE and POST phases** of both models.

3. **report.md**  
   A short experimental report including:
   - task description and target JSON schema
   - experimental setup (dataset, subset sizes, models, QLoRA configuration)
   - PRE vs POST evaluation tables
   - qualitative observations on hallucinations and schema compliance

In [ ]:
import csv
import json
from datetime import datetime
from pathlib import Path

out_dir = Path(".")
results_path = out_dir / "results.csv"
examples_path = out_dir / "examples.jsonl"
report_path = out_dir / "report.md"

# -------------------------
# Build results table
# -------------------------
def row_for(model_name: str, phase: str, metrics: dict):
    return {
        "model": model_name,
        "phase": phase,
        "valid_json_rate": float(metrics["valid_json_rate"]),
        "schema_ok_rate": float(metrics["schema_ok_rate"]),
        "ingredient_f1": float(metrics["ingredient_f1"]),
        "cooking_time_acc": float(metrics["cooking_time_acc"]),
        "temperature_acc": float(metrics["temperature_acc"]),
    }

rows = []
rows.append(row_for("qwen2.5-3b-instruct", "pre", qwen3b_pre_metrics))
rows.append(row_for("qwen2.5-3b-instruct", "post", qwen3b_post_metrics))
rows.append(row_for("mistral-7b-instruct-v0.3", "pre", m7b_pre_metrics))
rows.append(row_for("mistral-7b-instruct-v0.3", "post", m7b_post_metrics))

# deltas
def delta(pre: dict, post: dict):
    return {
        "valid_json_rate": post["valid_json_rate"] - pre["valid_json_rate"],
        "schema_ok_rate": post["schema_ok_rate"] - pre["schema_ok_rate"],
        "ingredient_f1": post["ingredient_f1"] - pre["ingredient_f1"],
        "cooking_time_acc": post["cooking_time_acc"] - pre["cooking_time_acc"],
        "temperature_acc": post["temperature_acc"] - pre["temperature_acc"],
    }

qwen_delta = delta(qwen3b_pre_metrics, qwen3b_post_metrics)
m7b_delta = delta(m7b_pre_metrics, m7b_post_metrics)

rows.append({
    "model": "qwen2.5-3b-instruct",
    "phase": "delta(post-pre)",
    **{k: float(v) for k, v in qwen_delta.items()}
})
rows.append({
    "model": "mistral-7b-instruct-v0.3",
    "phase": "delta(post-pre)",
    **{k: float(v) for k, v in m7b_delta.items()}
})

# Write CSV
fieldnames = ["model", "phase", "valid_json_rate", "schema_ok_rate", "ingredient_f1", "cooking_time_acc", "temperature_acc"]
with results_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in rows:
        w.writerow(r)

print("Wrote:", results_path)

# -------------------------
# Write examples.jsonl
# -------------------------
def dump_examples(tag: str, examples: list):
    """
    Each example in our lists is expected to be a dict with:
      - idx
      - input_text
      - raw
      - pred
      - label
      - meta (optional)
    """
    out = []
    for ex in examples:
        out.append({
            "tag": tag,
            "idx": ex.get("idx"),
            "input_text": ex.get("input_text"),
            "raw": ex.get("raw"),
            "pred": ex.get("pred"),
            "label": ex.get("label"),
            "meta": ex.get("meta", None),
        })
    return out

all_examples = []
all_examples += dump_examples("qwen3b_pre", qwen3b_pre_examples)
all_examples += dump_examples("qwen3b_post", qwen3b_post_examples)
all_examples += dump_examples("mistral7b_pre", m7b_pre_examples)
all_examples += dump_examples("mistral7b_post", m7b_post_examples)

with examples_path.open("w", encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Wrote:", examples_path, "| total examples:", len(all_examples))

# -------------------------
# Write report.md
# -------------------------
schema_block = """{
  "ingredients": [string, ...],
  "cooking_time": integer (minutes) or null,
  "temperature": {"value": integer, "unit": "C" or "F"} or null
}"""

def md_table_for(model_name: str, pre: dict, post: dict):
    keys = ["valid_json_rate", "schema_ok_rate", "ingredient_f1", "cooking_time_acc", "temperature_acc"]
    lines = []
    lines.append("| metric | PRE | POST | delta |")
    lines.append("|---|---:|---:|---:|")
    for k in keys:
        dv = post[k] - pre[k]
        lines.append(f"| {k} | {pre[k]:.4f} | {post[k]:.4f} | {dv:+.4f} |")
    return "\n".join(lines)

report_lines = []
report_lines.append("# Progetto 1.2 — Structured Extraction")
report_lines.append("")
report_lines.append("## Task")
report_lines.append("Dato un testo di ricetta in linguaggio naturale, il modello deve estrarre un output JSON con campi predefiniti.")
report_lines.append("")
report_lines.append("Schema JSON target:")
report_lines.append("```json")
report_lines.append(schema_block)
report_lines.append("```")
report_lines.append("")
report_lines.append("Vincoli principali:")
report_lines.append("- JSON sintatticamente valido")
report_lines.append("- Nessuna chiave extra (solo i 3 campi richiesti)")
report_lines.append("- Riduzione delle allucinazioni: se un campo non è presente nel testo, deve essere null (o [] per ingredients)")
report_lines.append("")
report_lines.append("## Setup sperimentale")
report_lines.append(f"- Dataset: SandhyaKilari/RecipeNLG_dataset (split train)")
report_lines.append(f"- Subset rapido: TRAIN_SIZE={TRAIN_SIZE}, EVAL_SIZE={EVAL_SIZE}, seed={SEED}")
report_lines.append("- 2 Small Language Models: Qwen2.5-3B-Instruct e Mistral-7B-Instruct-v0.3")
report_lines.append("- Valutazione PRE fine-tuning: generazione deterministica (do_sample=False, temperature=0) + parsing/validazione robusta")
report_lines.append("- Fine-tuning: QLoRA 4-bit (NF4), LoRA r=8 alpha=16, max_steps limitato per esecuzione rapida su Colab T4")
report_lines.append("- Valutazione POST fine-tuning: stessa procedura e stesso eval set")
report_lines.append("")
report_lines.append("## Metriche")
report_lines.append("- valid_json_rate: percentuale di output parsabili come JSON")
report_lines.append("- schema_ok_rate: percentuale senza chiavi extra e senza chiavi mancanti")
report_lines.append("- ingredient_f1: F1 su ingredienti (match case-insensitive dopo normalizzazione)")
report_lines.append("- cooking_time_acc: accuratezza su cooking_time (minuti), ignorando casi null")
report_lines.append("- temperature_acc: accuratezza su temperatura (value+unit), ignorando casi null")
report_lines.append("")
report_lines.append("## Risultati")
report_lines.append("")
report_lines.append("### Qwen2.5-3B-Instruct")
report_lines.append(md_table_for("qwen2.5-3b-instruct", qwen3b_pre_metrics, qwen3b_post_metrics))
report_lines.append("")
report_lines.append("### Mistral-7B-Instruct-v0.3")
report_lines.append(md_table_for("mistral-7b-instruct-v0.3", m7b_pre_metrics, m7b_post_metrics))
report_lines.append("")
report_lines.append("## Osservazioni qualitative")
report_lines.append("- Dopo fine-tuning entrambi i modelli migliorano sensibilmente su ingredienti, cooking_time e temperature.")
report_lines.append("- Qwen 3B mantiene sempre JSON valido e schema conforme; Mistral 7B POST ha una lieve riduzione (0.975) dovuta a pochi output non perfettamente parsabili o con testo extra.")
report_lines.append("- Gli esempi qualitativi (PRE/POST) sono salvati in examples.jsonl per ispezione manuale.")
report_lines.append("")
report_lines.append("## File prodotti")
report_lines.append("- results.csv")
report_lines.append("- examples.jsonl")
report_lines.append("- report.md")
report_lines.append("")

report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Wrote:", report_path)

Wrote: results.csv
Wrote: examples.jsonl | total examples: 32
Wrote: report.md


# Download Project Artifacts (and Optional Adapter Archive)

In this section we prepare the **final project artifacts for download**.

Files produced in the previous step:
- `results.csv`
- `report.md`
- `examples.jsonl`

The cell verifies that these files exist and prints their sizes.

Optionally, we also create a compressed archive:

- `adapters.zip`

This archive contains the **fine-tuned QLoRA adapters**:
- `qwen3b_adapter`
- `mistral7b_adapter`

Saving the adapters is optional, but useful if you want to:
- reproduce the experiments later
- share the trained adapters
- attach them to the project submission.

If the notebook is running in **Google Colab**, the files will be automatically downloaded to your local machine.

In [ ]:
from pathlib import Path
import zipfile

files_to_download = ["results.csv", "report.md", "examples.jsonl"]

for fn in files_to_download:
    p = Path(fn)
    if p.exists():
        print("OK:", fn, "| size:", p.stat().st_size, "bytes")
    else:
        print("MISSING:", fn)

# Optional: zip adapters (small, quick)
adapters_zip = Path("adapters.zip")
with zipfile.ZipFile(adapters_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ["qwen3b_adapter", "mistral7b_adapter"]:
        fp = Path(folder)
        if not fp.exists():
            print("Skip missing adapter folder:", folder)
            continue
        for f in fp.rglob("*"):
            if f.is_file():
                z.write(f, arcname=str(Path(folder) / f.relative_to(fp)))

print("Wrote:", adapters_zip)

# Colab download (works only in Colab)
try:
    from google.colab import files
    for fn in files_to_download:
        if Path(fn).exists():
            files.download(fn)
    if adapters_zip.exists():
        files.download(str(adapters_zip))
except Exception as e:
    print("Not running in Colab download environment:", type(e).__name__)

OK: results.csv | size: 566 bytes
OK: report.md | size: 2553 bytes
OK: examples.jsonl | size: 42436 bytes
Wrote: adapters.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>